# MLflow tracking

# Cell 1: connect and list experiments

In [1]:
import mlflow, pandas as pd
from mlflow.tracking import MlflowClient
import warnings; warnings.filterwarnings("ignore")

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
client = MlflowClient()

for e in client.search_experiments():
    print(e.experiment_id, e.name)

2 Future_Price_Regression
1 Good_Investment_Classification
0 Default


# Cell 2: compare classification runs

In [2]:
CLF_EXP = "Good_Investment_Classification"
runs = mlflow.search_runs(experiment_names=[CLF_EXP], order_by=["metrics.f1 DESC"])
runs = runs[runs["status"] == "FINISHED"]

cols = ["tags.mlflow.runName", "metrics.accuracy", "metrics.precision",
        "metrics.recall", "metrics.f1", "metrics.roc_auc"]
clf_table = runs[cols].rename(columns=lambda c: c.split(".")[-1]).round(4)
clf_table

,runName,accuracy,precision,recall,f1,roc_auc
0,clf_XGBoost_all_features,0.9983,0.9984,0.9983,0.9984,1.0000
1,clf_XGBoost_all_features,0.9983,0.9984,0.9983,0.9984,1.0000
2,clf_XGBoost_all_features,0.9983,0.9984,0.9983,0.9984,1.0000
3,clf_XGBoost_all_features,0.9983,0.9984,0.9983,0.9984,1.0000
9,clf_RandomForest_all_features,0.9961,0.9966,0.9962,0.9964,0.9995
10,clf_RandomForest_all_features,0.9961,0.9966,0.9962,0.9964,0.9995
11,clf_RandomForest_all_features,0.9961,0.9966,0.9962,0.9964,0.9995
13,clf_LogisticRegression_all_features,0.8875,0.8948,0.8952,0.8950,0.9625
14,clf_LogisticRegression_all_features,0.8875,0.8948,0.8952,0.8950,0.9625
15,clf_LogisticRegression_all_features,0.8875,0.8948,0.8952,0.8950,0.9625


# Cell 3: compare regression runs

In [3]:
REG_EXP = "Future_Price_Regression"
runs = mlflow.search_runs(experiment_names=[REG_EXP], order_by=["metrics.rmse ASC"])
runs = runs[runs["status"] == "FINISHED"]

cols = ["tags.mlflow.runName", "metrics.rmse", "metrics.mae", "metrics.r2"]
reg_table = runs[cols].rename(columns=lambda c: c.split(".")[-1]).round(4)
reg_table

,runName,rmse,mae,r2
0,reg_XGBoost_all_features,1.3887,1.0868,1.0000
1,reg_LinearRegression_all_features,11.3554,8.4711,0.9972
2,reg_RandomForest_all_features,15.2354,10.4310,0.9950
3,reg_XGBoost_no_price_features,214.7302,185.4247,0.0019


# Cell 4: view the Model Registry

In [4]:
for rm in client.search_registered_models():
    print(rm.name)
    for v in client.search_model_versions(f"name='{rm.name}'"):
        print("   version", v.version, "| run_id", v.run_id[:8], "| aliases", v.aliases)

RealEstate_Investment_Classifier
   version 1 | run_id b983869c | aliases []
RealEstate_Price_Regressor
   version 1 | run_id b6c86054 | aliases []


# Cell 5: mark the best versions as

In [5]:
client.set_registered_model_alias("RealEstate_Investment_Classifier", "champion", 1)
client.set_registered_model_alias("RealEstate_Price_Regressor", "champion", 1)

client.set_registered_model_tag("RealEstate_Investment_Classifier", "task", "classification")
client.set_registered_model_tag("RealEstate_Price_Regressor", "task", "regression")

# Cell 6: load the registered model and test it

In [6]:
import mlflow.sklearn

TRUSTED = ["sklearn.tree._tree.Tree", "xgboost.core.Booster",
           "xgboost.sklearn.XGBClassifier", "xgboost.sklearn.XGBRegressor"]

df = pd.read_csv("../data/processed_housing.csv")

clf_cols = __import__("joblib").load("../models/clf_columns.joblib")
sample = df[clf_cols["num"] + clf_cols["cat"]].head(5)

try:
    model = mlflow.sklearn.load_model("models:/RealEstate_Investment_Classifier@champion",
                                      skops_trusted_types=TRUSTED)
except TypeError:
    model = mlflow.sklearn.load_model("models:/RealEstate_Investment_Classifier@champion")

print("Predicted:", model.predict(sample))
print("Actual   :", df["Good_Investment"].head(5).values)

Predicted: [0 1 1 0 1]
Actual   : [0 1 1 0 1]


# Cell 7: tune the classifier with nested runs

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
import itertools

num_f, cat_f = clf_cols["num"], clf_cols["cat"]
# the no-price variant is the meaningful one to tune, since the full model is already ~99.8%
num_f = [c for c in num_f if c not in ["Price_in_Lakhs", "Price_per_SqFt_calc", "SqFt_per_BHK"]]

X, y = df[num_f + cat_f], df["Good_Investment"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

mlflow.set_experiment("Classifier_Tuning_NoPrice")
grid = list(itertools.product([3, 6, 9], [0.05, 0.1], [200]))

with mlflow.start_run(run_name="xgb_grid_search") as parent:
    best_f1 = 0
    for depth, lr, n in grid:
        with mlflow.start_run(run_name=f"depth{depth}_lr{lr}", nested=True):
            pre = ColumnTransformer([("num", StandardScaler(), num_f),
                                     ("cat", OneHotEncoder(handle_unknown="ignore"), cat_f)])
            pipe = Pipeline([("prep", pre),
                             ("model", XGBClassifier(max_depth=depth, learning_rate=lr, n_estimators=n,
                                                     eval_metric="logloss", n_jobs=-1, random_state=42))])
            pipe.fit(Xtr, ytr)
            pred, proba = pipe.predict(Xte), pipe.predict_proba(Xte)[:, 1]
            m = {"accuracy": accuracy_score(yte, pred), "f1": f1_score(yte, pred),
                 "roc_auc": roc_auc_score(yte, proba)}
            mlflow.log_params({"max_depth": depth, "learning_rate": lr, "n_estimators": n})
            mlflow.log_metrics(m)
            best_f1 = max(best_f1, m["f1"])
            print(f"depth={depth} lr={lr}  ->", {k: round(v, 4) for k, v in m.items()})
    mlflow.log_metric("best_f1", best_f1)

2026/11/01 19:33:02 INFO mlflow.tracking.fluent: Experiment with name 'Classifier_Tuning_NoPrice' does not exist. Creating a new experiment.


depth=3 lr=0.05  -> {'accuracy': 0.7059, 'f1': 0.7333, 'roc_auc': 0.8193}
depth=3 lr=0.1  -> {'accuracy': 0.7053, 'f1': 0.7301, 'roc_auc': 0.8194}
depth=6 lr=0.05  -> {'accuracy': 0.7069, 'f1': 0.7294, 'roc_auc': 0.82}
depth=6 lr=0.1  -> {'accuracy': 0.7076, 'f1': 0.7304, 'roc_auc': 0.82}
depth=9 lr=0.05  -> {'accuracy': 0.7107, 'f1': 0.7322, 'roc_auc': 0.8187}
depth=9 lr=0.1  -> {'accuracy': 0.7053, 'f1': 0.7268, 'roc_auc': 0.8154}
